In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  4787k      0  0:00:17  0:00:17 --:--:-- 9827k


In [2]:
!rm -rf /content/aclImdb/train/unsup

In [3]:
import os, pathlib, shutil, random

base_dir  = pathlib.Path('aclImdb')
val_dir   = base_dir / 'val'
train_dir = base_dir / 'train'

for category in ('neg', 'pos'):
  os.makedirs(val_dir / category)
  files  = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_files = int(0.2 * len(files))
  val_files = files[-num_val_files:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)

In [4]:
from tensorflow import keras
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    '/content/aclImdb/train', batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    '/content/aclImdb/val', batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    '/content/aclImdb/test', batch_size=batch_size)

for inputs, targets in train_ds:
  print("Inputs shape: ", inputs.shape)
  print("Inputs dtype: ", inputs.dtype)
  print("Targets shape: ", targets.shape)
  print("Targets dtype: ", targets.dtype)
  print("Inputs first sample: ", inputs[0])
  print("Targets first sample: ", targets[0])
  break

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
Inputs shape:  (32,)
Inputs dtype:  <dtype: 'string'>
Targets shape:  (32,)
Targets dtype:  <dtype: 'int32'>
Inputs first sample:  tf.Tensor(b"Lily Mars, a smalltown girl living in Indiana, dreams of making it big on Broadway and her aspirations are given a lift when successful Broadway producer John Thornway returns to his hometown for a visit. Lily tries everything she can to get Thornway to notice her, but he just gets annoyed with her antics. When Thornway goes back to New York to stage his show, Lily follows (unknown to John of course) and Thornway eventually gives her a small role in his next show, only as a favor to her family, however Thornway starts to fall for this young girl and a romance blossoms, which makes the show's leading lady, Isabel Rekay, jealous. When Isabel gets fed up with the John-Lily romance causing friction with the show, she leaves, a

## Unigram experimentation
---




In [5]:
from tensorflow.keras.layers import TextVectorization

text_vectorization = TextVectorization(
    max_tokens=20000,
    output_mode='multi_hot'
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_1gram_train_ds = train_ds.map(
    lambda x, y : (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_val_ds = val_ds.map(
    lambda x, y : (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_test_ds = test_ds.map(
    lambda x, y : (text_vectorization(x), y),
    num_parallel_calls=4)

for inputs, targets in binary_1gram_train_ds:
  print("Input shape: ", inputs.shape)
  print("Input dtype: ", inputs.dtype)
  print("Targets shape: ", targets.shape)
  print("Targets dtype: ", targets.dtype)
  print("Inputs first sample ", inputs[0])
  print("Targets first sample ", targets[0])
  break

Input shape:  (32, 20000)
Input dtype:  <dtype: 'int64'>
Targets shape:  (32,)
Targets dtype:  <dtype: 'int32'>
Inputs first sample  tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
Targets first sample  tf.Tensor(0, shape=(), dtype=int32)


In [6]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(num_tokens=20000, hidden_dim=16):
  inputs  = keras.Input(shape=(num_tokens,))
  x       = layers.Dense(hidden_dim, activation='relu')(inputs)
  x       = layers.Dropout(0.5)(x)
  outputs = layers.Dense(1, activation='sigmoid')(x)
  model   = keras.Model(inputs, outputs)

  model.compile(optimizer='rmsprop',
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

In [ ]:
model = get_model()
model.summary()

callbacks = keras.callbacks.ModelCheckpoint('binary_1gram.keras',
                                            save_best_only=True)

model.fit(binary_1gram_train_ds.cache(),
          validation_data=binary_1gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model('binary_1gram.keras')
print(f'Test acc {model.evaluate(binary_1gram_test_ds)[1]:.3f}')

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.7720 - loss: 0.4882 - val_accuracy: 0.8850 - val_loss: 0.2897
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9009 - loss: 0.2669 - val_accuracy: 0.8894 - val_loss: 0.2944
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9201 - loss: 0.2304 - val_accuracy: 0.8830 - val_loss: 0.3213
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9264 - loss: 0.2097 - val_accuracy: 0.8894 - val_loss: 0.3310
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9308 - loss: 0.2040 - val_accuracy: 0.8874 - val_loss: 0.3513
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9360 - loss: 0.1994 - val_accuracy: 0.8848 - val_loss: 0.3601
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9395 - loss: 0.1927 - val_accuracy: 0.8818 - val_loss: 0.3820
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.9397 - loss: 0.1973 - val_accuracy: 

## Bigram Experimentation
---

In [ ]:
text_vectorization_2 = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='multi_hot'
)

text_vectorization_2.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y : (text_vectorization_2(x), y),
    num_parallel_calls=4)

binary_2gram_val_ds = val_ds.map(
    lambda x, y : (text_vectorization_2(x), y),
    num_parallel_calls=4)

binary_2gram_test_ds = test_ds.map(
    lambda x, y : (text_vectorization_2(x), y),
    num_parallel_calls=4)

for inputs, targets in binary_2gram_train_ds:
  print("Input shape: ", inputs.shape)
  print("Input dtype: ", inputs.dtype)
  print("Targets shape: ", targets.shape)
  print("Targets dtype: ", targets.dtype)
  print("Inputs first sample ", inputs[0])
  print("Targets first sample ", targets[0])
  break

Input shape:  (32, 20000)
Input dtype:  <dtype: 'int64'>
Targets shape:  (32,)
Targets dtype:  <dtype: 'int32'>
Inputs first sample  tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
Targets first sample  tf.Tensor(0, shape=(), dtype=int32)


In [ ]:
model = get_model()
model.summary()

callbacks = keras.callbacks.ModelCheckpoint('binary_2gram.keras',
                                            save_best_only=True)

model.fit(binary_2gram_train_ds.cache(),
          validation_data=binary_2gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model('binary_2gram.keras')
print(f'Test acc {model.evaluate(binary_2gram_test_ds)[1]:.3f}')

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.8023 - loss: 0.4428 - val_accuracy: 0.8960 - val_loss: 0.2726
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9179 - loss: 0.2338 - val_accuracy: 0.8954 - val_loss: 0.2824
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9352 - loss: 0.1944 - val_accuracy: 0.8936 - val_loss: 0.3079
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9458 - loss: 0.1756 - val_accuracy: 0.8924 - val_loss: 0.3250
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9502 - loss: 0.1604 - val_accuracy: 0.8902 - val_loss: 0.3516
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9529 - loss: 0.1675 - val_accuracy: 0.8876 - val_loss: 0.3689
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9566 - loss: 0.1616 - val_accuracy: 0.8884 - val_loss: 0.3846
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9571 - loss: 0.1667 - val_accuracy:

## Bigrams with TF-IDF Encodings
---

In [10]:
tf_idf_text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='tf_idf'
)

tf_idf_text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y : (tf_idf_text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_val_ds = val_ds.map(
    lambda x, y : (tf_idf_text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_test_ds = test_ds.map(
    lambda x, y : (tf_idf_text_vectorization(x), y),
    num_parallel_calls=4)

model = get_model()
model.summary()

callbacks = keras.callbacks.ModelCheckpoint('tfidf_2gram.keras',
                                            save_best_only=True)

model.fit(tfidf_2gram_train_ds.cache(),
          validation_data=tfidf_2gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model('tfidf_2gram.keras')
print(f'Test acc {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}')

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 16s 24ms/step - accuracy: 0.7447 - loss: 0.7021 - val_accuracy: 0.8940 - val_loss: 0.2768
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8920 - loss: 0.2919 - val_accuracy: 0.9060 - val_loss: 0.2850
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9114 - loss: 0.2415 - val_accuracy: 0.9002 - val_loss: 0.3058
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9154 - loss: 0.2221 - val_accuracy: 0.8960 - val_loss: 0.3240
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9209 - loss: 0.2095 - val_accuracy: 0.8938 - val_loss: 0.3474
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9239 - loss: 0.1990 - val_accuracy: 0.8848 - val_loss: 0.3425
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9267 - loss: 0.1848 - val_accuracy: 0.8884 - val_loss: 0.3464
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9271 - loss: 0.1899 - val_accuracy: 

# Sequence Models
---
## Biderectional LSTM's
---

In [16]:
import tensorflow as tf

max_length = 600
max_tokens = 20000
lstm_text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_length)

lstm_text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y : (lstm_text_vectorization(x), y),
    num_parallel_calls=4)

int_val_ds = val_ds.map(
    lambda x, y : (lstm_text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y : (lstm_text_vectorization(x), y),
    num_parallel_calls=4)

inputs   = keras.Input(shape=(None,), dtype='int64')
embedded = layers.Embedding(input_dim=max_tokens, output_dim=16)
x        = layers.Bidirectional(layers.LSTM(32))(embedded)
x        = layers.Dropout(0.5)(x)
outputs  = layers.Dense(1, activation='sigmoid')(x)

model    = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: <Embedding name=embedding, built=False> (of type <class 'keras.src.layers.core.embedding.Embedding'>)

In [ ]:
callbacks = keras.callbacks.ModelCheckpoint('one_hot_bidir_listm.keras',
                                            save_best_only=True)

model.fit(int_train_ds.cache(),
          validation_data=int_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model('one_hot_bidir_listm.keras')
print(f'Test acc {model.evaluate(int_test_ds)[1]:.3f}')